# VisionX V4 Final — Frozen Backbone Fine-Tuning (21 Kelas)
Notebook ini disiapkan untuk melatih model **VisionX V4 Final (21 kelas)** menggunakan GPU T4 di Google Colab.

### Inovasi Kunci V4-Final (Solusi Catastrophic Forgetting):
1. **`freeze=10`**: Membekukan seluruh layer backbone CSPDarknet (layer 0–9). Fitur visual V3 untuk kelas `person`, `laptop`, `cup`, `bottle` **terkunci permanen dan aman dari degradasi**.
2. **`lr0=0.001`**: Learning rate lebih lembut untuk fine-tuning leher deteksi (FPN/PAN) dan head baru tanpa *gradient shock*.
3. **Dataset 21 Kelas Lengkap (`processed_v3.zip`)**: Mempertahankan seluruh 7 kelas baru (`helm`, `tanpa_helm`, `car`, `motorcycle`, dll.).

### Checklist Persiapan:
File di Google Drive Anda **sudah ada dari sesi sebelumnya** (tidak perlu upload ulang dataset besar!):
- `processed_v3.zip` (Dataset 15.097 citra)
- `visionx_v3_best.pt` (Checkpoint bobot model V3)

Cukup upload notebook ini ke Google Colab, pilih **T4 GPU**, lalu klik **Run All**!

### Cell 1: Hubungkan Google Drive & Install Ultralytics

In [ ]:
from google.colab import drive
import os, sys

# 1. Hubungkan Google Drive
drive.mount('/content/drive')

# 2. Install Ultralytics YOLO
!pip install -q ultralytics

import torch
print('\n--- Environment Check ---')
print('PyTorch Version:', torch.__version__)
print('CUDA Available :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device     :', torch.cuda.get_device_name(0))
    vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f'VRAM           : {vram:.2f} GB')
else:
    print('PERINGATAN: GPU belum aktif! Silakan buka Runtime > Change runtime type > Pilih T4 GPU.')


### Cell 2: Ekstrak Dataset processed_v3 dari Google Drive

In [ ]:
import zipfile, os, yaml

# Cari processed_v3.zip di Drive
ZIP_PATH = '/content/drive/MyDrive/processed_v3.zip'
if not os.path.exists(ZIP_PATH):
    for root, dirs, files in os.walk('/content/drive/MyDrive'):
        if 'processed_v3.zip' in files:
            ZIP_PATH = os.path.join(root, 'processed_v3.zip')
            break

print(f'Menggunakan dataset: {ZIP_PATH}')
DEST_DIR = '/content/datasets/processed_v3'
os.makedirs(DEST_DIR, exist_ok=True)

print('Mengekstrak dataset ke /content/datasets/processed_v3...')
with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
    zip_ref.extractall(DEST_DIR)

# Update path di dataset.yaml ke path absolut Colab
yaml_path = os.path.join(DEST_DIR, 'dataset.yaml')
with open(yaml_path, 'r') as f:
    cfg = yaml.safe_load(f)

cfg['path'] = DEST_DIR
with open(yaml_path, 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False, sort_keys=False)

print('Ekstraksi selesai!')
print(f'Train images : {len(os.listdir(os.path.join(DEST_DIR, "images/train")))}')
print(f'Val images   : {len(os.listdir(os.path.join(DEST_DIR, "images/val")))}')
print(f'Test images  : {len(os.listdir(os.path.join(DEST_DIR, "images/test")))}')
print(f'Total Classes: {len(cfg["names"])}')


### Cell 3: Load Checkpoint Base VisionX V3 (Transfer Learning)

In [ ]:
# Cari visionx_v3_best.pt di Drive
MODEL_PATH = '/content/drive/MyDrive/visionx_v3_best.pt'
if not os.path.exists(MODEL_PATH):
    for root, dirs, files in os.walk('/content/drive/MyDrive'):
        if 'visionx_v3_best.pt' in files:
            MODEL_PATH = os.path.join(root, 'visionx_v3_best.pt')
            break

print(f'Checkpoint model ditemukan: {MODEL_PATH}')
from ultralytics import YOLO
model = YOLO(MODEL_PATH)
print(f'Model V3 ter-load dengan {len(model.names)} kelas awal.')


### Cell 4: Training VisionX V4 Final (freeze=10, lr0=0.001 di GPU T4)

In [ ]:
import time
t0 = time.time()

# Konfigurasi V4-Final dengan Frozen Backbone
results = model.train(
    data='/content/datasets/processed_v3/dataset.yaml',
    epochs=20,
    patience=6,
    batch=16,
    imgsz=640,
    device=0,
    optimizer='auto',
    lr0=0.001,       # Learning rate fine-tuning halus
    lrf=0.01,
    freeze=10,       # MEMBEKUKAN BACKBONE CSPDarknet (layer 0-9) agar fitur V3 tidak terlupakan!
    project='runs/detect',
    name='visionx_v4_final',
    exist_ok=True,
    verbose=True,
    seed=42
)

t_total = time.time() - t0
print(f'\nTraining selesai dalam {t_total/60:.2f} menit ({t_total/20:.1f} detik/epoch)!')


### Cell 5: Evaluasi Otomatis 21 Kelas (Val & Test Splits) + Komparasi V3

In [ ]:
from pathlib import Path
from ultralytics import YOLO

best_final_path = '/content/runs/detect/runs/detect/visionx_v4_final/weights/best.pt'
if not os.path.exists(best_final_path):
    candidates = list(Path('/content').glob('**/visionx_v4_final/weights/best.pt'))
    if candidates:
        best_final_path = str(candidates[0])

print(f'Menggunakan bobot terbaik: {best_final_path}')
model_final = YOLO(best_final_path)

V3_BASELINE_MAP50 = {
    0: 72.2, 1: 49.6, 2: 54.1, 3: 70.2, 4: 94.9, 5: 44.9,
    6: 86.8, 7: 99.5, 8: 98.5, 9: 89.9, 10: 99.5, 11: 99.5,
    12: 0.0, 13: 0.0
}

def evaluate_and_print(split_name):
    print('\n' + '=' * 105)
    print(f' EVALUASI V4-FINAL - SPLIT: {split_name.upper()}')
    print('=' * 105)
    metrics = model_final.val(data='/content/datasets/processed_v3/dataset.yaml', split=split_name, imgsz=640, device=0, verbose=False)
    print(f'Overall Precision : {metrics.box.mp*100:.2f}%')
    print(f'Overall Recall    : {metrics.box.mr*100:.2f}%')
    print(f'Overall mAP50     : {metrics.box.map50*100:.2f}%')
    print(f'Overall mAP50-95  : {metrics.box.map*100:.2f}%')
    print('-' * 105)
    print(f'{"ID":2} | {"Class Name":14} | {"Instances":9} | {"Precision":10} | {"Recall":8} | {"mAP50":8} | {"mAP50-95":9} | {"V3 mAP50":8} | {"Delta":7} | {"Status"}')
    print('-' * 105)
    
    ap_classes = metrics.box.ap_class_index
    for i, c in enumerate(ap_classes):
        p, r, ap50, ap95 = metrics.box.class_result(i)
        inst = int(metrics.nt_per_class[c]) if metrics.nt_per_class is not None else 0
        p_pct = p * 100
        r_pct = r * 100
        ap50_pct = ap50 * 100
        ap95_pct = ap95 * 100
        
        v3_str = f'{V3_BASELINE_MAP50[c]:.1f}%' if c in V3_BASELINE_MAP50 else 'N/A (New)'
        if c in V3_BASELINE_MAP50 and V3_BASELINE_MAP50[c] > 0:
            diff = ap50_pct - V3_BASELINE_MAP50[c]
            diff_str = f'{diff:+.1f}%'
            status = 'REGRESI' if diff < -10.0 else ('Turun Ringan' if diff < -3.0 else ('NAIK' if diff > 3.0 else 'Stabil'))
        elif c in [12, 13]:
            diff_str = '-'
            status = '0 Sampel'
        else:
            diff_str = 'Baru'
            status = 'Kelas Baru'
        print(f'{c:2d} | {model_final.names[c]:14s} | {inst:9d} | {p_pct:9.1f}% | {r_pct:7.1f}% | {ap50_pct:7.1f}% | {ap95_pct:8.1f}% | {v3_str:8s} | {diff_str:7s} | {status}')
    print('-' * 105)

evaluate_and_print('val')
evaluate_and_print('test')


### Cell 6: Simpan Hasil Training Kembali ke Google Drive

In [ ]:
import os, shutil
from pathlib import Path

run_dir = '/content/runs/detect/runs/detect/visionx_v4_final'
if not os.path.exists(run_dir):
    candidates = list(Path('/content').glob('**/visionx_v4_final'))
    if candidates:
        run_dir = str(candidates[0])

out_zip = '/content/drive/MyDrive/visionx_v4_final_results.zip'
print(f'Mengompres {run_dir} ke {out_zip}...')
!zip -q -r "{out_zip}" "{run_dir}"

if os.path.exists(out_zip):
    print(f'BERHASIL! File tersimpan di Google Drive: {out_zip} ({os.path.getsize(out_zip)/1024/1024:.2f} MB)')
    print('Silakan download visionx_v4_final_results.zip ke laptop Anda!')
else:
    print('Gagal menyimpan zip ke Google Drive.')
